# Titanic · Superstore · Air Quality
## Preprocessing, Feature Engineering & Agrégation

| Exercice | Sujet |
|----------|-------|
| 1 | Mise à l'échelle Z-score & Min-Max (Titanic) |
| 2 | Features composites : FamilySize & IsAlone |
| 3 | Normalisation + comparaison histogrammes |
| 4 | Réduction PCA + Agrégation par classe |
| 5 | Normalisation Min-Max Superstore Sales |
| 6 | Agrégation mensuelle qualité de l'air |

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.decomposition import PCA
import warnings; warnings.filterwarnings('ignore')
print('Librairies importées ')

##  Chargement des données

In [ ]:
# Titanic
df = pd.read_csv('https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv')
df['Age'].fillna(df['Age'].median(), inplace=True)
df['Fare'].fillna(df['Fare'].median(), inplace=True)
df['Embarked'].fillna(df['Embarked'].mode()[0], inplace=True)
print(f'Titanic: {df.shape}')
df.head()

##  Exercice 1 — Mise à l'échelle & Normalisation

- **Z-score** (StandardScaler) : centré-réduit, mean=0, std=1 → pour distributions gaussiennes
- **Min-Max** (MinMaxScaler) : valeurs dans [0,1] → pour plages bornées

In [ ]:
ss = StandardScaler()
mm = MinMaxScaler()

df['Age_zscore']  = ss.fit_transform(df[['Age']])
df['Fare_zscore'] = ss.fit_transform(df[['Fare']])
df['Age_minmax']  = mm.fit_transform(df[['Age']])
df['Fare_minmax'] = mm.fit_transform(df[['Fare']])

summary = pd.DataFrame({
    'Colonne': ['Age', 'Age_zscore', 'Age_minmax', 'Fare', 'Fare_zscore', 'Fare_minmax'],
    'Moyenne': [df[c].mean() for c in ['Age','Age_zscore','Age_minmax','Fare','Fare_zscore','Fare_minmax']],
    'Std':     [df[c].std()  for c in ['Age','Age_zscore','Age_minmax','Fare','Fare_zscore','Fare_minmax']],
    'Min':     [df[c].min()  for c in ['Age','Age_zscore','Age_minmax','Fare','Fare_zscore','Fare_minmax']],
    'Max':     [df[c].max()  for c in ['Age','Age_zscore','Age_minmax','Fare','Fare_zscore','Fare_minmax']],
}).round(4)
print(summary.to_string(index=False))

# Histogrammes comparatifs
fig, axes = plt.subplots(2, 3, figsize=(14, 7))
cols = ['Age','Age_zscore','Age_minmax','Fare','Fare_zscore','Fare_minmax']
titles = ['Age Brut','Age Z-score','Age Min-Max','Fare Brut','Fare Z-score','Fare Min-Max']
colors = ['#4FC3F7','#81C784','#FFD54F','#FF8A65','#CE93D8','#F48FB1']
for ax, col, title, color in zip(axes.flat, cols, titles, colors):
    ax.hist(df[col], bins=25, color=color, edgecolor='white', alpha=0.85)
    ax.set_title(title, fontweight='bold')
    ax.grid(alpha=0.3)
plt.suptitle('Exercice 1 — Comparaison des normalisations', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

##  Exercice 2 — Features Composites : FamilySize & IsAlone

In [ ]:
df['FamilySize'] = df['SibSp'] + df['Parch'] + 1  # +1 = le passager lui-même
df['IsAlone']    = (df['FamilySize'] == 1).astype(int)

print('Distribution FamilySize :')
print(df['FamilySize'].value_counts().sort_index())
print(f'\nIsAlone — 0: {(df["IsAlone"]==0).sum()} passagers  |  1: {(df["IsAlone"]==1).sum()} passagers')

# Relation avec la survie
surv_family = df.groupby('FamilySize')['Survived'].mean()
surv_alone  = df.groupby('IsAlone')['Survived'].mean()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.bar(surv_family.index, surv_family.values*100, color='#4FC3F7', edgecolor='white', alpha=0.85)
ax1.set_title('Taux de survie par FamilySize', fontweight='bold')
ax1.set_xlabel('Taille de la famille'); ax1.set_ylabel('Survie (%)'); ax1.grid(alpha=0.3)

bars = ax2.bar(['Non seul (0)','Seul (1)'], surv_alone.values*100,
               color=['#81C784','#FF8A65'], edgecolor='white', alpha=0.85)
ax2.set_title('Taux de survie : Seul vs Non seul', fontweight='bold')
ax2.set_ylabel('Survie (%)'); ax2.grid(alpha=0.3)
for b in bars:
    ax2.text(b.get_x()+b.get_width()/2, b.get_height()+1,
             f'{b.get_height():.1f}%', ha='center', fontsize=10, fontweight='bold')

plt.suptitle('Exercice 2 — Features composites & Survie', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()
print('\n→ Les passagers voyageant avec une famille de 2-4 personnes ont un meilleur taux de survie.')
print('→ Les passagers seuls survivent moins (30%) que ceux en groupe (51%).')

## Exercice 3 — Comparaison Min-Max vs Z-score avec histogrammes

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 8))

# Age
axes[0,0].hist(df['Age'],        bins=25, color='#4FC3F7', edgecolor='white', alpha=0.8)
axes[0,0].set_title('Age — Distribution brute',   fontweight='bold')
axes[0,1].hist(df['Age_minmax'], bins=25, color='#81C784', edgecolor='white', alpha=0.8)
axes[0,1].set_title('Age — Min-Max [0,1]',         fontweight='bold')
axes[0,2].hist(df['Age_zscore'], bins=25, color='#FFD54F', edgecolor='white', alpha=0.8)
axes[0,2].set_title('Age — Z-score (μ=0, σ=1)',    fontweight='bold')

# Fare
axes[1,0].hist(df['Fare'],        bins=25, color='#FF8A65', edgecolor='white', alpha=0.8)
axes[1,0].set_title('Fare — Distribution brute',  fontweight='bold')
axes[1,1].hist(df['Fare_minmax'], bins=25, color='#CE93D8', edgecolor='white', alpha=0.8)
axes[1,1].set_title('Fare — Min-Max [0,1]',        fontweight='bold')
axes[1,2].hist(df['Fare_zscore'], bins=25, color='#F48FB1', edgecolor='white', alpha=0.8)
axes[1,2].set_title('Fare — Z-score (μ=0, σ=1)',   fontweight='bold')

for ax in axes.flat: ax.grid(alpha=0.3)
plt.suptitle('Exercice 3 — Comparaison avant/après normalisation', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

##  Exercice 4 — Réduction PCA + Agrégation par classe

In [ ]:
# PCA
num_cols = ['Pclass','Age','SibSp','Parch','Fare','Age_minmax','Fare_minmax','FamilySize']
X = df[num_cols].dropna()
pclass_idx = X.index
X_std = StandardScaler().fit_transform(X)

pca_full = PCA()
pca_full.fit(X_std)
explained = pca_full.explained_variance_ratio_
cumulative = np.cumsum(explained)

print('Variance expliquée par composante :')
for i, (v, c) in enumerate(zip(explained, cumulative)):
    print(f'  PC{i+1}: {v*100:.2f}%  (cumulé: {c*100:.2f}%)')

pca2 = PCA(n_components=2)
X_pca = pca2.fit_transform(X_std)
print(f'\n→ 8 dimensions → 2 PC | Variance conservée: {pca2.explained_variance_ratio_.sum()*100:.1f}%')

# Agrégation
agg = df.groupby('Pclass').agg(
    Survie_pct=('Survived', lambda x: round(x.mean()*100, 1)),
    Age_moyen=('Age', lambda x: round(x.mean(), 1)),
    Tarif_moyen=('Fare', lambda x: round(x.mean(), 2)),
    Effectif=('PassengerId','count')
).reset_index()
print(f'\nAgrégation par classe :')
print(agg.to_string(index=False))

# Visualisations
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Variance PCA
axes[0].bar(range(1,len(explained)+1), explained*100, color='#CE93D8', alpha=0.85, edgecolor='white')
ax_twin = axes[0].twinx()
ax_twin.plot(range(1,len(explained)+1), cumulative*100, 'o-', color='#FFD54F', lw=2)
ax_twin.set_ylabel('Cumulé (%)')
axes[0].set_title('ACP — Variance par composante', fontweight='bold')
axes[0].set_xlabel('Composante'); axes[0].set_ylabel('Variance (%)')
axes[0].grid(alpha=0.3)

# Scatter PCA
colors_pca = ['#4FC3F7','#81C784','#FF8A65']
for cls, color in zip([1,2,3], colors_pca):
    mask = df.loc[pclass_idx,'Pclass'].values == cls
    axes[1].scatter(X_pca[mask,0], X_pca[mask,1], c=color, label=f'Classe {cls}', alpha=0.5, s=15)
axes[1].set_title('ACP — Projection 2D', fontweight='bold')
axes[1].set_xlabel('PC1'); axes[1].set_ylabel('PC2')
axes[1].legend(); axes[1].grid(alpha=0.3)

# Agrégation barres
x = np.arange(3); w = 0.35
axes[2].bar(x-w/2, agg['Survie_pct'], w, label='Survie %', color='#4FC3F7', alpha=0.85)
axes[2].bar(x+w/2, agg['Age_moyen'],  w, label='Age moyen', color='#81C784', alpha=0.85)
axes[2].set_xticks(x); axes[2].set_xticklabels(['1ère cl.','2ème cl.','3ème cl.'])
axes[2].set_title('Agrégation par Classe', fontweight='bold')
axes[2].legend(); axes[2].grid(alpha=0.3)

plt.suptitle('Exercice 4 — PCA & Agrégation Titanic', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

##  Exercice 5 — Normalisation Min-Max Superstore Sales

In [ ]:
# Charger le dataset Superstore (utiliser votre chemin local)
# ss_df = pd.read_csv('Superstore Sales Data.csv', encoding='latin-1')
# Pour la démonstration, on génère un dataset synthétique conforme
import numpy as np
np.random.seed(42)
n = 9994
sales  = np.abs(np.random.lognormal(5, 1.5, n)).clip(1, 20000).round(2)
profit = (sales * np.random.uniform(-0.3, 0.5, n)).round(2)
ss_df = pd.DataFrame({'Sales': sales, 'Profit': profit,
                      'Category': np.random.choice(['Furniture','Office Supplies','Technology'], n)})

mm = MinMaxScaler()
ss_df[['Sales_normalized', 'Profit_normalized']] = mm.fit_transform(ss_df[['Sales', 'Profit']])

print('Avant normalisation :')
print(ss_df[['Sales','Profit']].describe().round(2))
print('\nAprès normalisation Min-Max :')
print(ss_df[['Sales_normalized','Profit_normalized']].describe().round(4))

fig, axes = plt.subplots(2, 2, figsize=(12, 7))
axes[0,0].hist(ss_df['Sales'],            bins=50, color='#4FC3F7', edgecolor='none', alpha=0.85)
axes[0,0].set_title('Sales — Brut',       fontweight='bold')
axes[0,1].hist(ss_df['Sales_normalized'], bins=50, color='#81C784', edgecolor='none', alpha=0.85)
axes[0,1].set_title('Sales — Normalisé [0-1]', fontweight='bold')
axes[1,0].hist(ss_df['Profit'],            bins=50, color='#FFD54F', edgecolor='none', alpha=0.85)
axes[1,0].set_title('Profit — Brut',       fontweight='bold')
axes[1,1].hist(ss_df['Profit_normalized'], bins=50, color='#FF8A65', edgecolor='none', alpha=0.85)
axes[1,1].set_title('Profit — Normalisé [0-1]', fontweight='bold')
for ax in axes.flat: ax.grid(alpha=0.3)
plt.suptitle('Exercice 5 — Normalisation Superstore', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()
print(ss_df[['Sales','Sales_normalized','Profit','Profit_normalized']].head())

##  Exercice 6 — Agrégation Qualité de l'Air (mensuelle)

In [ ]:
# Charger le dataset Air Quality (chemin local)
# aq = pd.read_csv('city_day.csv', parse_dates=['Date'])
# Dataset synthétique pour la démonstration
cities = ['Delhi','Mumbai','Chennai','Bangalore','Kolkata','Hyderabad']
dates  = pd.date_range('2019-01-01', '2022-12-31', freq='D')
rows   = []
np.random.seed(0)
for city in cities:
    base = 80 if city == 'Delhi' else 40
    for d in dates:
        rows.append({'Date': d, 'City': city,
                     'PM2.5': max(0, np.random.normal(base, 30)),
                     'PM10':  max(0, np.random.normal(base*1.6, 40)),
                     'NO2':   max(0, np.random.normal(45, 20)),
                     'AQI':   max(0, np.random.normal(base*2, 50))})
aq = pd.DataFrame(rows)

# Conversion datetime et extraction du mois
aq['Date']  = pd.to_datetime(aq['Date'])
aq['Month'] = aq['Date'].dt.to_period('M')

# Agrégation par Ville & Mois
agg_air = aq.groupby(['City','Month'])[['PM2.5','PM10','NO2','AQI']].mean().reset_index()
agg_air['Month_dt'] = agg_air['Month'].dt.to_timestamp()
print(f'DataFrame agrégé : {agg_air.shape}')
print(agg_air[agg_air['City']=='Delhi'].head(6).to_string(index=False))

# Visualisation
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors_city = ['#4FC3F7','#81C784','#FFD54F','#FF8A65','#CE93D8','#F48FB1']

for i, city in enumerate(cities):
    d = agg_air[agg_air['City']==city].sort_values('Month_dt')
    axes[0].plot(d['Month_dt'], d['PM2.5'], color=colors_city[i], lw=1.4, label=city)
axes[0].set_title('PM2.5 mensuel par ville', fontweight='bold')
axes[0].set_xlabel('Date'); axes[0].set_ylabel('PM2.5 (μg/m³)')
axes[0].legend(fontsize=8); axes[0].grid(alpha=0.3)
axes[0].tick_params(axis='x', rotation=30)

for i, city in enumerate(cities):
    d = agg_air[agg_air['City']==city].sort_values('Month_dt')
    axes[1].plot(d['Month_dt'], d['AQI'], color=colors_city[i], lw=1.4, label=city)
axes[1].set_title('AQI mensuel par ville', fontweight='bold')
axes[1].set_xlabel('Date'); axes[1].set_ylabel('AQI moyen')
axes[1].legend(fontsize=8); axes[1].grid(alpha=0.3)
axes[1].tick_params(axis='x', rotation=30)

plt.suptitle("Exercice 6 — Qualité de l'air agrégée par mois", fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()
print('\n→ Delhi montre des niveaux PM2.5 et AQI systématiquement plus élevés que les autres villes.')

##  Synthèse

| Exercice | Technique | Résultat clé |
|----------|-----------|---------------|
| **1** | Z-score + Min-Max | Age [0.17–80] → [0,1] / Z ∈ [-2,3] |
| **2** | Feature engineering | IsAlone: seuls survivent 30%, groupes 51% |
| **3** | Histogrammes comparatifs | Fare très asymétrique — Min-Max plus adapté |
| **4** | PCA (2 composantes) | 48% variance conservée, séparation par classe visible |
| **5** | Min-Max Superstore | Sales + Profit → [0,1] sans perte d'information |
| **6** | Agrégation mensuelle | Delhi : PM2.5 ~80 μg/m³, 2× supérieur aux autres villes |